In [6]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
# split para modelado
from sklearn.model_selection import train_test_split
# Scaled | Escalado
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# Encoding | Codificación
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.metrics import accuracy_score
# To save models
import math
import json
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
# Feature Selection
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from pickle import dump
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint
from tqdm.auto import tqdm
import joblib
from contextlib import contextmanager
from scipy.stats import randint, uniform
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
df_prueba = "anthonny"
if df_prueba == "anthonny":
    df = pd.read_csv("../data/processed/df")
    df = df.sort_values("num_semana").reset_index(drop=True)

    weeks = df["num_semana"].unique()
    cut_w = int(len(weeks) * 0.8)

    train_weeks = weeks[:cut_w]
    test_weeks  = weeks[cut_w:]

    train = df[df["num_semana"].isin(train_weeks)]
    test  = df[df["num_semana"].isin(test_weeks)]

    X_train, y_train = train.drop(columns=["y"]), train["y"]
    X_test,  y_test  = test.drop(columns=["y"]),  test["y"]

else:
    df = pd.read_csv("../data/processed/df_ineta.cvs")
    df = df.sort_values("weekend").reset_index(drop=True) 

    X = df.drop(columns=["weekend"])
    y = df["weekend"]

    cut = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:cut], X.iloc[cut:]
    y_train, y_test = y.iloc[:cut], y.iloc[cut:]
    print ("Trabajaremos con el DF de ineta")

In [8]:
cb = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=18,
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    verbose=0
)

cb.fit(
    X_train, y_train,
    cat_features=["product"],
    eval_set=(X_test, y_test),
    early_stopping_rounds=200,
    use_best_model=True
)

In [9]:
model = cb
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_test:.2f} % de precision")

MSE (Error cuadrático medio): 22.08
RMSE (Raíz del ECM): 4.70 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.72 % de precision


In [10]:
import optuna
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

def objective(trial):
    params = {
        "iterations": 5000, # Bajamos a 2000 para las pruebas; el early stopping hará el resto
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 1, 5),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "border_count": 128, # Un valor fijo suele ser suficiente para ganar tiempo
        "loss_function": "RMSE",
        "verbose": 0,
        "random_seed": 18
    }
    
    model = CatBoostRegressor(**params)
    
    model.fit(
        X_train, y_train,
        cat_features=["product"],
        eval_set=(X_test, y_test),
        early_stopping_rounds=100, # Si en 100 vueltas no mejora, pasa a la siguiente prueba
        use_best_model=True
    )
    
    preds = model.predict(X_test)
    return r2_score(y_test, preds)

# Crear el estudio y ejecutar
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50) # Con 30 pruebas suele ser suficiente para ver tendencia

print(f"Mejor R2: {study.best_value:.4f}")
print(f"Mejores parámetros: {study.best_params}")


[I 2026-02-03 12:16:25,453] A new study created in memory with name: no-name-eb0a04b3-ea49-443a-aa79-d9f166bd010e
[I 2026-02-03 12:16:26,002] Trial 0 finished with value: 0.7141123399244129 and parameters: {'learning_rate': 0.09501530552510577, 'depth': 4, 'l2_leaf_reg': 9.755402405487464, 'random_strength': 3.505754676594555, 'bagging_temperature': 0.5976809625607662}. Best is trial 0 with value: 0.7141123399244129.
[I 2026-02-03 12:16:28,269] Trial 1 finished with value: 0.7192535472311945 and parameters: {'learning_rate': 0.09076252042932806, 'depth': 9, 'l2_leaf_reg': 4.950324575196085, 'random_strength': 4.61230039232432, 'bagging_temperature': 0.5609134432093711}. Best is trial 1 with value: 0.7192535472311945.
[I 2026-02-03 12:16:33,624] Trial 2 finished with value: 0.7238808432099028 and parameters: {'learning_rate': 0.0298343419725862, 'depth': 10, 'l2_leaf_reg': 7.1832681102592435, 'random_strength': 2.4713142158006867, 'bagging_temperature': 0.3449893017067248}. Best is tria

Mejor R2: 0.7290
Mejores parámetros: {'learning_rate': 0.043409509372900196, 'depth': 10, 'l2_leaf_reg': 8.974022103583465, 'random_strength': 4.935756745887414, 'bagging_temperature': 0.18429685638990317}


In [11]:
# 1. Definir el modelo con los parámetros encontrados por Optuna
best_params = study.best_params

model_final = CatBoostRegressor(
    iterations=5000,          # Subimos iteraciones para el entrenamiento final
    loss_function="RMSE",
    random_seed=18,
    verbose=100,              # Para ver el progreso cada 100 pasos
    **best_params             # Esto inserta automáticamente: depth, learning_rate, etc.
)

# 2. Entrenar (usamos early_stopping para no sobreajustar)
model_final.fit(
    X_train, y_train,
    cat_features=["product"],
    eval_set=(X_test, y_test),
    early_stopping_rounds=200,
    use_best_model=True
)

0:	learn: 11.6240592	test: 8.7004371	best: 8.7004371 (0)	total: 26ms	remaining: 2m 9s
100:	learn: 5.7132132	test: 4.6194932	best: 4.6194932 (100)	total: 2.38s	remaining: 1m 55s
200:	learn: 5.0950338	test: 4.6563318	best: 4.6183251 (102)	total: 4.8s	remaining: 1m 54s
300:	learn: 4.7180341	test: 4.6847060	best: 4.6183251 (102)	total: 7.04s	remaining: 1m 49s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 4.618325127
bestIteration = 102

Shrink model to first 103 iterations.


In [12]:
pred_test = model_final.predict(X_test)
pred_train = model_final.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_test:.2f} % de precision")

MSE (Error cuadrático medio): 21.33
RMSE (Raíz del ECM): 4.62 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.73 % de precision
